In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

/mnt/c/Users/xie_j/Documents/llm-zoomcamp-code/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
# Using RAGVector class defined in the previous session:

from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [3]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. If you want a certificate, make sure you submit your project while submissions are still open.'

In [4]:
vs_index.close()

In [ ]:
"""
Here is how the two compare:
* minsearch VectorSearch: in-memory (numpy), exact cosine similarity, must re-compute embeddings on startup, 
  good for experiments and notebooks
* sqlitesearch VectorSearchIndex: persistent (SQLite .db file), ANN (LSH/IVF/HNSW) with exact rerank, 
  can open an existing index, good for projects and persistence
"""